In [0]:
from pyspark.sql.functions import col, when, round

is_df = spark.read.table("workspace.stock_data.income_statements_silver")

bs_df = spark.read.table("workspace.stock_data.balance_sheets_silver")

bs_df = bs_df.withColumn(
    'debt_to_equity_ratio', 
    round(col('total_debt') / col('total_equity'), 2)
)

bs_df = bs_df.withColumn(
    "current_ratio",
    round(when(col("total_current_liabilities") == 0, 0)
    .otherwise(col("total_current_assets") / col("total_current_liabilities")),2)
)

bs_df = bs_df.select(
    'symbol', 'fiscal_year', 'period', 'total_assets', 'total_liabilities', 'total_equity', 'total_debt', 'total_current_assets', 'total_current_liabilities', 'debt_to_equity_ratio', 'current_ratio'
)

financials_merged = is_df.join(
    bs_df, 
    on=['symbol', 'fiscal_year', 'period'], 
    how='left'
)

financials_merged = financials_merged.withColumn(
    'roe', 
    round(col('net_income') / col('total_equity'), 2)
)

financials_merged.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.financials_gold')


In [0]:
%sql
SELECT *
FROM workspace.stock_data.financials_gold